In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os

def detect_changes(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    """
    두 이미지에서 객체를 검출하고 변화(추가/제거)를 감지합니다.
    """
    # 모델 로드
    model = YOLO(model_path)
    
    # 이미지 읽기
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    if image1 is None or image2 is None:
        print(f"Error: 이미지 로드 실패 - {image1_path}, {image2_path}")
        return
    
    # 객체 검출
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    
    # 객체 정보 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    
    # 변화 비교
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)
    
    # 결과 분석 출력
    analyze_results(objects1, objects2, added, removed)

def extract_objects(results, confidence_threshold):
    """
    YOLO 결과에서 confidence threshold 이상인 객체 추출.
    """
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def calculate_iou(box1, box2):
    """
    두 bounding box의 IoU 계산.
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    if union == 0:
        return 0
    return intersection / union

def compare_objects(objects1, objects2, iou_threshold):
    """
    두 이미지의 객체 리스트를 비교하여 추가/제거 객체를 반환.
    """
    added = []
    removed = []
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold and obj1['class_id'] == obj2['class_id']:
                matched1.add(i)
                matched2.add(j)
                break
        if i not in matched1:
            removed.append(obj1)
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    return added, removed

def visualize_changes(image1, image2, added, removed):
    """
    변화(추가/제거) 객체를 이미지에 표시하여 시각화.
    """
    # 두 이미지를 좌우로 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 (오른쪽 이미지, 초록색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 (왼쪽 이미지, 빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    plt.figure(figsize=(15, 7))
    plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    plt.title('Changes Detected (Left: Image1, Right: Image2)')
    plt.axis('off')
    plt.show()

def analyze_results(objects1, objects2, added, removed):
    """
    변화 감지 결과 및 클래스별 통계 출력.
    """
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

def process_directory(directory_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    """
    디렉토리 내 연속 이미지 쌍에 대해 변화 감지 수행.
    """
    image_files = [f for f in os.listdir(directory_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    image_files.sort()
    for i in range(len(image_files) - 1):
        image1_path = os.path.join(directory_path, image_files[i])
        image2_path = os.path.join(directory_path, image_files[i+1])
        print(f"\nProcessing: {image_files[i]} and {image_files[i+1]}")
        detect_changes(image1_path, image2_path, model_path, confidence_threshold, iou_threshold)

# 사용 예시
if __name__ == "__main__":
    image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
    image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
    model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'
    
    # 이미지 두 장 비교
    detect_changes(image1_path, image2_path, model_path)
    
    # 디렉토리 내 모든 이미지 연속 쌍 비교
    # directory_path = 'D:/LHJ-Capstone Wire detection/wireod'
    # process_directory(directory_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

def detect_changes(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    
    # YOLO를 사용하여 객체 탐지
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    
    # 객체 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    
    # 객체 비교
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    
    # 결과 시각화
    visualize_changes(image1, image2, added, removed)
    
    # 결과 분석
    analyze_results(objects1, objects2, added, removed)

def extract_objects(results, confidence_threshold):
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'


detect_changes(image1_path, image2_path, model_path)


In [ ]:
## Optical Flow

import cv2
import numpy as np
from ultralytics import YOLO
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

def detect_changes_with_optical_flow(image1_path, image2_path, model_path, confidence_threshold=0.5, iou_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image1 = cv2.imread(image1_path)
    image2 = cv2.imread(image2_path)
    
    # YOLO를 사용하여 객체 탐지
    results1 = model(image1)[0]
    results2 = model(image2)[0]
    
    # 객체 추출
    objects1 = extract_objects(results1, confidence_threshold)
    objects2 = extract_objects(results2, confidence_threshold)
    
    # Optical Flow를 사용한 객체 이동 추적
    track_points, status, err = track_objects_with_optical_flow(image1, image2, objects1)
    
    # 객체 비교
    added, removed = compare_objects(objects1, objects2, iou_threshold)
    
    # 결과 시각화
    visualize_changes(image1, image2, added, removed, track_points)
    
    # 결과 분석
    analyze_results(objects1, objects2, added, removed)

def extract_objects(results, confidence_threshold):
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

def track_objects_with_optical_flow(image1, image2, objects1):
    # 이전 프레임에서의 특징점 추출
    old_points = np.array([[(obj['bbox'][0] + obj['bbox'][2]) // 2, 
                            (obj['bbox'][1] + obj['bbox'][3]) // 2] for obj in objects1], dtype=np.float32)
    
    # Optical Flow 계산
    new_points, status, err = cv2.calcOpticalFlowPyrLK(cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY), 
                                                       cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY), 
                                                       old_points, None)
    
    return new_points, status, err

def compare_objects(objects1, objects2, iou_threshold):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def visualize_changes(image1, image2, added, removed, track_points=None):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # Optical Flow로 추적된 객체 표시 (파란색)
    if track_points is not None:
        for pt in track_points:
            if pt is not None:
                x, y = pt.ravel()
                cv2.circle(result_image, (int(x) + width, int(y)), 5, (255, 0, 0), -1)
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def analyze_results(objects1, objects2, added, removed):
    print(f"Image 1: {len(objects1)} objects detected")
    print(f"Image 2: {len(objects2)} objects detected")
    print(f"Added objects: {len(added)}")
    print(f"Removed objects: {len(removed)}")
    
    class_counts1 = {}
    class_counts2 = {}
    
    for obj in objects1:
        class_counts1[obj['class_id']] = class_counts1.get(obj['class_id'], 0) + 1
    
    for obj in objects2:
        class_counts2[obj['class_id']] = class_counts2.get(obj['class_id'], 0) + 1
    
    print("\nClass distribution in Image 1:")
    for class_id, count in class_counts1.items():
        print(f"Class {class_id}: {count}")
    
    print("\nClass distribution in Image 2:")
    for class_id, count in class_counts2.items():
        print(f"Class {class_id}: {count}")

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'


detect_changes_with_optical_flow(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os
from typing import List, Dict, Tuple

def detect_objects(image_path: str, model_path: str, confidence_threshold: float = 0.5) -> List[Dict]:
    """YOLO 모델을 사용하여 이미지에서 객체 검출"""
    model = YOLO(model_path)
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"이미지를 찾을 수 없습니다: {image_path}")
    
    results = model(image)[0]
    objects = []
    
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id),
                'center': [(x1+x2)//2, (y1+y2)//2]
            })
    return objects

def calculate_iou(box1: List[int], box2: List[int]) -> float:
    """두 바운딩 박스 간 IoU 계산"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union != 0 else 0.0

def compare_objects(
    objects1: List[Dict], 
    objects2: List[Dict], 
    iou_threshold: float = 0.5
) -> Tuple[List[Dict], List[Dict]]:
    """두 프레임 간 객체 비교하여 추가/제거된 객체 반환"""
    added, removed = [], []
    matched1, matched2 = set(), set()

    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            if (calculate_iou(obj1['bbox'], obj2['bbox']) > iou_threshold and 
                obj1['class_id'] == obj2['class_id']):
                matched1.add(i)
                matched2.add(j)
                break
        if i not in matched1:
            removed.append(obj1)

    added = [obj for j, obj in enumerate(objects2) if j not in matched2]
    return added, removed

def track_with_optical_flow(
    prev_image: np.ndarray, 
    curr_image: np.ndarray, 
    prev_objects: List[Dict]
) -> Tuple[np.ndarray, np.ndarray]:
    """광류 기반 객체 이동 추적"""
    prev_gray = cv2.cvtColor(prev_image, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_image, cv2.COLOR_BGR2GRAY)
    
    # 초기 포인트 설정
    prev_pts = np.array([obj['center'] for obj in prev_objects], dtype=np.float32)
    
    # 광류 계산
    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, prev_pts, None, winSize=(15,15), maxLevel=3
    )
    
    return curr_pts, status

def visualize_results(
    image1: np.ndarray, 
    image2: np.ndarray, 
    added: List[Dict], 
    removed: List[Dict], 
    track_points: np.ndarray
) -> None:
    """변화 결과 시각화 (Matplotlib 사용)"""
    combined = np.hstack((image1, image2))
    width = image1.shape[1]
    
    # 트랙 포인트 표시
    for pt in track_points:
        x, y = pt.astype(int)
        cv2.circle(combined, (x + width, y), 5, (255,0,0), -1)
    
    # 추가/제거 객체 표시
    for obj in added:
        x1,y1,x2,y2 = obj['bbox']
        cv2.rectangle(combined, (x1+width,y1), (x2+width,y2), (0,255,0), 2)
        cv2.putText(combined, 'Added', (x1+width, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    
    for obj in removed:
        x1,y1,x2,y2 = obj['bbox']
        cv2.rectangle(combined, (x1,y1), (x2,y2), (0,0,255), 2)
        cv2.putText(combined, 'Removed', (x1, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
    
    # Matplotlib 출력
    plt.figure(figsize=(20,10))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title("Object Tracking with Optical Flow")
    plt.axis('off')
    plt.show()

def main_workflow(
    image1_path: str, 
    image2_path: str, 
    model_path: str, 
    confidence: float = 0.5, 
    iou_thresh: float = 0.4
) -> None:
    """메인 워크플로우"""
    # 객체 검출
    prev_objects = detect_objects(image1_path, model_path, confidence)
    curr_objects = detect_objects(image2_path, model_path, confidence)
    
    # 변화 분석
    added, removed = compare_objects(prev_objects, curr_objects, iou_thresh)
    
    # 광류 추적
    prev_img = cv2.imread(image1_path)
    curr_img = cv2.imread(image2_path)
    track_pts, status = track_with_optical_flow(prev_img, curr_img, prev_objects)
    
    # 유효한 트랙 포인트 필터링
    valid_pts = track_pts[status.squeeze() == 1]
    
    # 결과 시각화
    visualize_results(prev_img, curr_img, added, removed, valid_pts)
    
    # 결과 출력
    print(f"추가 객체: {len(added)}개")
    print(f"제거 객체: {len(removed)}개")
    print(f"추적된 포인트: {len(valid_pts)}/{len(track_pts)}")

if __name__ == "__main__":
    image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
    image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
    model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'
    
    main_workflow(image1_path, image2_path, model_path)


In [ ]:
#YOLO + IOU (2024.09.03)
import cv2
import numpy as np  # numpy 임포트 추가
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    iou = intersection / float(area1 + area2 - intersection)
    return iou

def compare_objects(objects1, objects2, iou_threshold=0.5):
    added = []
    removed = []
    
    matched1 = set()
    matched2 = set()
    
    for i, obj1 in enumerate(objects1):
        for j, obj2 in enumerate(objects2):
            if i in matched1 or j in matched2:
                continue
            
            iou = calculate_iou(obj1['bbox'], obj2['bbox'])
            if iou > iou_threshold:
                matched1.add(i)
                matched2.add(j)
                break
        
        if i not in matched1:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'


image1 = cv2.imread(image1_path)
image2 = cv2.imread(image2_path)

objects1 = detect_objects(image1_path, model_path)
objects2 = detect_objects(image2_path, model_path)

added, removed = compare_objects(objects1, objects2)

visualize_changes(image1, image2, added, removed)



In [ ]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms as transforms

# YOLO 모델 로드
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'
model = YOLO(model_path)

# 객체 탐지 함수
def detect_objects(image, model, confidence_threshold=0.5):
    results = model(image)[0]
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    return objects

# 특징 벡터 추출기 정의 (간단한 CNN 예제)
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc = nn.Linear(256 * 32 * 32, 512)  # assuming input image size 128x128

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(-1, 256 * 32 * 32)  # Flatten
        x = torch.relu(self.fc(x))
        return x

# 두 이미지 간의 차이 계산
def compute_difference(image1, image2, model, feature_extractor):
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ])

    image1 = transform(image1).unsqueeze(0)
    image2 = transform(image2).unsqueeze(0)

    # 모델을 사용하여 특징 벡터 추출
    feature1 = feature_extractor(image1)
    feature2 = feature_extractor(image2)

    # 유클리드 거리로 두 특징 간의 차이를 계산
    difference = torch.norm(feature1 - feature2, p=2, dim=1)
    return difference.item()

# 모델 초기화
feature_extractor = FeatureExtractor()

# 이미지 로드 및 객체 탐지
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'

image1 = Image.open(image1_path).convert('RGB')
image2 = Image.open(image2_path).convert('RGB')

# YOLO를 사용한 객체 탐지
objects1 = detect_objects(image1, model)
objects2 = detect_objects(image2, model)

# 차이 계산
difference = compute_difference(image1, image2, model, feature_extractor)

print(f"Difference between images: {difference}")

# 차이가 특정 임계값을 넘으면 변화를 감지했다고 판단
threshold = 5.0
if difference > threshold:
    print("Change detected")
else:
    print("No significant change detected")


In [ ]:
#SURF 알고리즘

import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        patch = image[y1:y2, x1:x2]
        patches.append((patch, obj))
    return patches

def surf_feature_matching(patch1, patch2):
    # SURF 객체 생성 (임계값 400은 일반적으로 사용되는 값입니다)
    surf = cv2.xfeatures2d.SURF_create()
    
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = surf.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = surf.detectAndCompute(patch2, None)
    
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    
    # BFMatcher 객체 생성
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    
    return matches, keypoints1, keypoints2

def compare_objects_with_surf(image1, image2, objects1, objects2):
    added = []
    removed = []
    
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    
    matched1 = set()
    matched2 = set()
    
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            
            matches, _, _ = surf_feature_matching(patch1, patch2)
            
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_surf(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    
    # SURF를 사용한 객체 비교
    added, removed = compare_objects_with_surf(image1, image2, objects1, objects2)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_surf(image1_path, image2_path, model_path)


In [ ]:
#ORB

import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        patch = image[y1:y2, x1:x2]
        patches.append((patch, obj))
    return patches

def orb_feature_matching(patch1, patch2):
    # ORB 객체 생성
    orb = cv2.ORB_create()
    
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = orb.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = orb.detectAndCompute(patch2, None)
    
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    
    # BFMatcher 객체 생성 (해밍 거리 사용)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    
    return matches, keypoints1, keypoints2

def compare_objects_with_orb(image1, image2, objects1, objects2):
    added = []
    removed = []
    
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    
    matched1 = set()
    matched2 = set()
    
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            
            matches, _, _ = orb_feature_matching(patch1, patch2)
            
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_orb(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    
    # ORB를 사용한 객체 비교
    added, removed = compare_objects_with_orb(image1, image2, objects1, objects2)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_orb(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image_path, model_path, confidence_threshold=0.5):
    # YOLO 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # YOLO를 사용하여 객체 탐지
    results = model(image)[0]
    
    # 객체 추출
    objects = []
    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = r
        if score > confidence_threshold:
            objects.append({
                'bbox': [int(x1), int(y1), int(x2), int(y2)],
                'score': score,
                'class_id': int(class_id)
            })
    
    return objects

def extract_object_patches(image, objects):
    patches = []
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        patch = image[y1:y2, x1:x2]
        patches.append((patch, obj))
    return patches

def surf_feature_matching(patch1, patch2):
    # SURF 객체 생성 (임계값 400은 일반적으로 사용되는 값입니다)
    surf = cv2.xfeatures2d.SURF_create(500)
    
    # 특징점 및 디스크립터 계산
    keypoints1, descriptors1 = surf.detectAndCompute(patch1, None)
    keypoints2, descriptors2 = surf.detectAndCompute(patch2, None)
    
    # 디스크립터가 None인 경우 처리
    if descriptors1 is None or descriptors2 is None:
        return [], keypoints1, keypoints2
    
    # BFMatcher 객체 생성
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    
    # 디스크립터 매칭
    matches = bf.match(descriptors1, descriptors2)
    
    # 매칭 결과 정렬
    matches = sorted(matches, key=lambda x: x.distance)
    
    return matches, keypoints1, keypoints2

def compare_objects_with_surf(image1, image2, objects1, objects2):
    added = []
    removed = []
    
    image1_patches = extract_object_patches(image1, objects1)
    image2_patches = extract_object_patches(image2, objects2)
    
    matched1 = set()
    matched2 = set()
    
    for i, (patch1, obj1) in enumerate(image1_patches):
        best_match = None
        best_match_idx = -1
        max_matches = 0
        
        for j, (patch2, obj2) in enumerate(image2_patches):
            if j in matched2:
                continue
            
            matches, _, _ = surf_feature_matching(patch1, patch2)
            
            if len(matches) > max_matches:
                max_matches = len(matches)
                best_match = obj2
                best_match_idx = j
        
        if best_match is not None and max_matches > 0:
            matched1.add(i)
            matched2.add(best_match_idx)
        else:
            removed.append(obj1)
    
    for j, obj2 in enumerate(objects2):
        if j not in matched2:
            added.append(obj2)
    
    return added, removed

def visualize_changes(image1, image2, added, removed):
    # 두 이미지를 나란히 붙임
    result_image = np.hstack((image1, image2))
    height, width = image1.shape[:2]
    
    # 추가된 객체 표시 (녹색)
    for obj in added:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1 + width, y1), (x2 + width, y2), (0, 255, 0), 2)
        cv2.putText(result_image, 'Added', (x1 + width, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 제거된 객체 표시 (빨간색)
    for obj in removed:
        x1, y1, x2, y2 = obj['bbox']
        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(result_image, 'Removed', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    # 결과 표시
    cv2.imshow('Changes Detected', result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def detect_changes_with_surf(image1_path, image2_path, model_path, confidence_threshold=0.5):
    # YOLO 모델을 사용하여 첫 번째 이미지의 객체 탐지
    image1 = cv2.imread(image1_path)
    objects1 = detect_objects(image1_path, model_path, confidence_threshold)
    
    # YOLO 모델을 사용하여 두 번째 이미지의 객체 탐지
    image2 = cv2.imread(image2_path)
    objects2 = detect_objects(image2_path, model_path, confidence_threshold)
    
    # SURF를 사용한 객체 비교
    added, removed = compare_objects_with_surf(image1, image2, objects1, objects2)
    
    # 변화 시각화
    visualize_changes(image1, image2, added, removed)

# 사용 예시
image1_path = 'D:/LHJ-Capstone Wire detection/wireod/test (5).jpg'
image2_path = 'D:/LHJ-Capstone Wire detection/wireod/test (6).jpg'
model_path = 'D:/LHJ-Capstone Wire detection/ultralytics-main/runs/segment/train8 best/weights/best.pt'

detect_changes_with_surf(image1_path, image2_path, model_path)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_and_visualize_wires_segmentation(image_path, model_path, confidence_threshold=0.5):
    # 모델 로드
    model = YOLO(model_path)
    
    # 이미지 로드
    image = cv2.imread(image_path)
    
    # 객체 탐지 및 세그멘테이션 수행
    results = model(image, stream=True)
    
    # 탐지된 와이어 시각화
    visualized_image = visualize_wires_segmentation(image, results, confidence_threshold)
    
    # 결과 표시
    cv2.imshow('Wire Segmentation', visualized_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # 결과 저장 (선택사항)
    cv2.imwrite('wire_segmentation_result.jpg', visualized_image)

def visualize_wires_segmentation(image, results, confidence_threshold):
    visualized_image = image.copy()
    
    for r in results:
        masks = r.masks
        if masks is None:
            continue
        
        for i, mask in enumerate(masks.data):
            if r.boxes.conf[i] > confidence_threshold and r.boxes.cls[i] == 2:  # 클래스 ID 0이 와이어라고 가정
                # 마스크를 이미지 크기에 맞게 조정
                mask = mask.cpu().numpy()
                mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
                
                # 마스크를 이진화
                mask = (mask > 0.5).astype(np.uint8)
                
                # 세그멘테이션 마스크에 색상 적용
                color_mask = np.zeros_like(image, dtype=np.uint8)
                color_mask[mask == 1] = [0, 255, 0]  # 녹색으로 와이어 표시
                
                # 원본 이미지와 색상 마스크 블렌딩
                alpha = 0.5  # 투명도 설정
                visualized_image = cv2.addWeighted(visualized_image, 1, color_mask, alpha, 0)
                
                # 신뢰도 점수 표시
                score = r.boxes.conf[i].item()
                label = f'Wire: {score:.2f}'
                cv2.putText(visualized_image, label, (int(r.boxes.xyxy[i][0]), int(r.boxes.xyxy[i][1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    return visualized_image

# 사용 예시
image_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 3.jpg'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train5/weights/best.pt'

detect_and_visualize_wires_segmentation(image_path, model_path, confidence_threshold=0.5)

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_and_visualize_objects(image_path, model_path, confidence_threshold=0.5):
    # Load model
    model = YOLO(model_path)
    
    # Load image
    image = cv2.imread(image_path)
    
    # Perform object detection and segmentation
    results = model(image, stream=True)
    
    # Visualize detected objects
    visualized_image = visualize_objects(image, results, confidence_threshold)
    
    # Display results
    cv2.imshow('Wire, Polo, and Cross Arm Segmentation', visualized_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # Save results (optional)
    cv2.imwrite('segmentation_result.jpg', visualized_image)

def visualize_objects(image, results, confidence_threshold):
    visualized_image = image.copy()
    
    class_colors = {
        0: [0, 0, 255],  # Blue for wire
        1: [255, 0, 0],  # Red for polo
        2: [0, 255, 0]   # Green for cross arm
    }
    
    class_names = {
        0: "Cross Arm",
        1: "Polo",
        2 : "Wire"
    }
    
    for r in results:
        masks = r.masks
        if masks is None:
            continue
        
        for i, mask in enumerate(masks.data):
            class_id = int(r.boxes.cls[i].item())
            if r.boxes.conf[i] > confidence_threshold and class_id in [0, 1, 2]:
                # Resize mask to match image size
                mask = mask.cpu().numpy()
                mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
                
                # Binarize mask
                mask = (mask > 0.5).astype(np.uint8)
                
                # Apply color to segmentation mask
                color_mask = np.zeros_like(image, dtype=np.uint8)
                color_mask[mask == 1] = class_colors[class_id]
                
                # Blend original image with color mask
                alpha = 0.5  # Set transparency
                visualized_image = cv2.addWeighted(visualized_image, 1, color_mask, alpha, 0)
                
                # Display confidence score
                score = r.boxes.conf[i].item()
                label = f'{class_names[class_id]}: {score:.2f}'
                cv2.putText(visualized_image, label, (int(r.boxes.xyxy[i][0]), int(r.boxes.xyxy[i][1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, class_colors[class_id], 2)
    
    return visualized_image

# Usage example
image_path = 'C:/Users/AI-LHJ/Downloads/source/ch5/test 5.jpg'
model_path = 'C:/Users/AI-LHJ/Desktop/ultralytics-main/runs/segment/train5/weights/best.pt'

detect_and_visualize_objects(image_path, model_path, confidence_threshold=0.5)